# Test q1_26 across a few as-of dates

Target quarter `q1_26` averaging windows (per `product_open_dates.txt`):
- **Y (cal)**: 2025-06-01 → 2025-12-20
- **Q**: 2025-10-01 → 2025-12-20
- **M1/M2/M3** (power & gas): 2025-12-01 → 2025-12-20
- **EUA M3**: 2025-10-01 → 2025-12-20

Sampled as-ofs probe each transition point.
In-memory only — no xlsx writes.

In [1]:
import os
from datetime import date
from pathlib import Path

import pandas as pd

# Read market data (futures.csv, the PFC, holidays.csv) from the repo's data/ rather than
# notebooks/data/: the package resolves it against the working directory.
os.environ["RO_DATA_DIR"] = str((Path("..") / "data").resolve())

from ro_calculation import load_price_frames, run_asof

pd.set_option("display.float_format", "{:.2f}".format)

TARGET_QUARTER = "q3_26"

df_power, df_gas, df_eua = load_price_frames()
summary_rows: list[dict] = []

def _run(asof: date) -> dict:
    """Run run_asof for the notebook's data/quarter and record the RO scenarios."""
    result = run_asof(asof, df_power, df_gas, df_eua, TARGET_QUARTER)
    summary_rows.append({"asof": asof, **result["ro_scenarios"]})
    return result

## 2025-08-01 — only Y window open, mid-stream

In [7]:
_run(date(2026, 6, 30)).keys()
# _run(date(2026, 6, 30))['run_as_of_status']

dict_keys(['run_as_of', 'run_as_of_status', 'ro_scenarios', 'legs', 'df'])

In [10]:
_run(date(2026, 6, 30))['legs']

{'power_Y': {'price_source': ['OMIP'], 'trading_days_remaining': 0},
 'power_Q': {'price_source': ['OMIP'], 'trading_days_remaining': 0},
 'power_M1': {'price_source': ['OMIP'], 'trading_days_remaining': 0},
 'power_M2': {'price_source': ['OMIP'], 'trading_days_remaining': 0},
 'power_M3': {'price_source': ['OMIP'], 'trading_days_remaining': 0},
 'gas_Cal': {'price_source': ['MIBGAS'], 'trading_days_remaining': 0},
 'gas_Q': {'price_source': ['MIBGAS'], 'trading_days_remaining': 0},
 'gas_M1': {'price_source': ['MIBGAS'], 'trading_days_remaining': 0},
 'gas_M2': {'price_source': ['MIBGAS'], 'trading_days_remaining': 0},
 'gas_M3': {'price_source': ['MIBGAS'], 'trading_days_remaining': 0},
 'EUA_M3': {'price_source': ['ICE'], 'trading_days_remaining': 0}}

## Exchange fallback: synthetic data check

Power's primary exchange is OMIP, gas's is MIBGAS, EUA's is ICE (`EXCHANGE_PRIORITY`).
Power falls back to EEX, gas falls back to EEX (PVB product only) when the primary
exchange has no data for a leg by `asof`. If neither exchange has data, the leg still
inherits from its parent (`M → Q → Y`), exactly as before.

Built here with small synthetic frames (not `data/futures.csv`) so each fallback path is
exercised deterministically:
- power `Y`/`Q`: only OMIP has data → primary, direct.
- power `M1`: only EEX (area `ES`) has data → fallback exchange, direct (the `area="DE"` row
  proves the area filter keeps foreign EEX rows out).
- power `M2`/`M3`: neither OMIP nor EEX has data → inherits from `Q` (OMIP).

In [3]:
from ro_calculation import gather_price_components

FALLBACK_TARGET_QUARTER = "q1_26"
FALLBACK_ASOF = date(2025, 12, 10)


def _row(commodity, exchange, area, product, strip, market_date, delivery_date, end_delivery_date, close, price_type=None):
    return {
        "commodity": commodity, "exchange": exchange, "area": area, "product": product,
        "strip": strip, "market_date": market_date, "delivery_date": delivery_date,
        "end_delivery_date": end_delivery_date, "close": close, "price_type": price_type,
    }


power_rows = []
# OMIP: Y and Q covered, but no M1/M2/M3 at all.
for d in ["2025-06-02", "2025-11-01", "2025-12-05"]:
    power_rows.append(_row("power", "OMIP", "ES", "Baseload", "cal", d, "2026-01-01", "2026-12-31", 60.0))
    power_rows.append(_row("power", "OMIP", "ES", "Baseload", "quarter", d, "2026-01-01", "2026-03-31", 65.0))
# EEX: M1 covered (fallback); M2/M3 not covered on either exchange.
for d in ["2025-12-02", "2025-12-08"]:
    power_rows.append(_row("power", "EEX", "ES", "Baseload", "month", d, "2026-01-01", "2026-01-31", 70.0))
power_rows.append(_row("power", "EEX", "DE", "Baseload", "month", "2025-12-08", "2026-01-01", "2026-01-31", 999.0))
df_power_fb = pd.DataFrame(power_rows)

# Minimal valid gas (MIBGAS) / EUA (ICE) data so gather_price_components has a full set to run.
# MIBGAS rows need price_type="reference" — MIBGAS carries a "last" series too (see README).
gas_rows = []
for d in ["2025-06-02", "2025-11-01", "2025-12-05", "2025-12-08"]:
    for strip, ds, de in [("cal", "2026-01-01", "2026-12-31"), ("quarter", "2026-01-01", "2026-03-31"),
                            ("month", "2026-01-01", "2026-01-31"), ("month", "2026-02-01", "2026-02-28"),
                            ("month", "2026-03-01", "2026-03-31")]:
        gas_rows.append(_row("gas", "MIBGAS", "ES", "Baseload", strip, d, ds, de, 30.0, price_type="reference"))
df_gas_fb = pd.DataFrame(gas_rows)

eua_rows = [_row("EUA", "ICE", "EU", "EUA", "quarter", d, "2026-01-01", "2026-03-31", 75.0)
            for d in ["2025-10-02", "2025-12-05"]]
df_eua_fb = pd.DataFrame(eua_rows)

components_fb = gather_price_components(df_power_fb, df_gas_fb, df_eua_fb, FALLBACK_TARGET_QUARTER, FALLBACK_ASOF)
power_fb = components_fb[components_fb["commodity"] == "power"].set_index("strip")
display(power_fb)

# price_source is a list of the distinct sources that contributed, e.g. ["OMIP", "PFC"].
assert power_fb.loc["Y", "price_source"] == ["OMIP"]
assert power_fb.loc["Q", "price_source"] == ["OMIP"]
assert power_fb.loc["M1", "price_source"] == ["EEX (fallback)"]
assert power_fb.loc["M1", "fixed"] == 70.0  # only the ES rows (70.0), not the DE noise row (999.0)
assert power_fb.loc["M2", "price_source"] == ["OMIP, propagated from Q"]
assert power_fb.loc["M3", "price_source"] == ["OMIP, propagated from Q"]
# "total" price_source now lists the distinct sources actually used across legs
# (in leg order, deduped) instead of collapsing to the generic "mixed" label.
assert power_fb.loc["total", "price_source"] == ["OMIP", "EEX (fallback)", "OMIP, propagated from Q"]
print("Exchange fallback checks passed.")

,quarter,commodity,fixed,float,reference,pct_open,price_source,primary_direct
strip,,,,,,,,
Y,q1_26,power,60.00,60.00,60.00,0.05,[OMIP],True
Q,q1_26,power,65.00,65.00,65.00,0.12,[OMIP],True
M1,q1_26,power,70.00,70.00,70.00,0.53,[EEX (fallback)],False
M2,q1_26,power,65.00,65.00,65.00,0.12,"[OMIP, propagated from Q]",False
M3,q1_26,power,65.00,65.00,65.00,0.12,"[OMIP, propagated from Q]",False
total,q1_26,power,64.50,64.50,64.50,0.17,"[OMIP, EEX (fallback), OMIP, propagated from Q]",False


Exchange fallback checks passed.


## Multiple quarters / multiple as-ofs in one `run_asof` call

`run_asof` accepts either `asof` or `target_quarter` (or both) as a list:
- scalar/scalar → same single result dict as before.
- one list → a dict keyed by the listed argument.
- both lists → a dict keyed by asof, each value a dict keyed by target_quarter.

In [16]:
# Multiple quarters, single as-of.
ASOF = date(2026, 7, 23)
QUARTERS = ["q1_27", "q2_27", "q3_27", "q4_27"]

multi_quarter = run_asof(ASOF, df_power, df_gas, df_eua, QUARTERS)
assert set(multi_quarter.keys()) == set(QUARTERS)
for q in QUARTERS:
    single = run_asof(ASOF, df_power, df_gas, df_eua, q)
    assert pd.Series(multi_quarter[q]["ro_scenarios"]).equals(pd.Series(single["ro_scenarios"]))

pd.DataFrame({q: multi_quarter[q]["ro_scenarios"] for q in QUARTERS}).T['reference'].mean()

np.float64(66.64906935627978)

In [5]:
# Multiple as-ofs, single quarter.
ASOFS = [date(2026, 6, 30), date(2026, 7, 1), date(2026, 7, 15)]

multi_asof = run_asof(ASOFS, df_power, df_gas, df_eua, TARGET_QUARTER)
assert set(multi_asof.keys()) == set(ASOFS)
for a in ASOFS:
    single = run_asof(a, df_power, df_gas, df_eua, TARGET_QUARTER)
    assert pd.Series(multi_asof[a]["ro_scenarios"]).equals(pd.Series(single["ro_scenarios"]))

pd.DataFrame({a: multi_asof[a]["ro_scenarios"] for a in ASOFS}).T

,fixed,float,reference
2026-06-30,55.53,NaN,55.53
2026-07-01,55.53,NaN,55.53
2026-07-15,55.53,NaN,55.53


## Full export: q1_26 to q4_31, asof 2026-07-24, use_pfc=True

24 quarters. 10 of them (`q3_28`, `q1_29`-`q3_29`, `q1_30`-`q3_30`, `q1_31`-`q3_31`) have
no EUA futures data at all this far out in `data/futures.csv` — ICE only quotes EUA
Dec-delivery contracts that far forward.

Previously this needed a notebook-level workaround (a synthetic EUA row averaged from a
separate daily-shaped-curve file) for exactly those 10 quarters. Now that
`_decompose_with_fallback` prefers a leg's own PFC window over inheriting a coarser
parent leg's price when no exchange has anything for that leg, `run_asof(...,
use_pfc=True)` alone covers all 24 quarters — see the `price_source` column below.

In [6]:
FULL_ASOF = date(2026, 7, 24)
FULL_QUARTERS = [f"q{q}_{y}" for y in range(26, 32) for q in range(1, 5)]

full_dfs = [run_asof(FULL_ASOF, df_power, df_gas, df_eua, q, use_pfc=True)["df"] for q in FULL_QUARTERS]
full_export_df = pd.concat(full_dfs, ignore_index=True)

full_out_path = Path("..") / "output" / f"ro_components_q1_26_to_q4_31_asof_{FULL_ASOF}.csv"
full_export_df.to_csv(full_out_path, index=False)
print(f"Wrote {len(full_export_df)} rows to {full_out_path}")
full_export_df[full_export_df["commodity"] == "EUA"][["quarter", "fixed", "float", "reference", "price_source"]]

Wrote 336 rows to ..\output\ro_components_q1_26_to_q4_31_asof_2026-07-24.csv


,quarter,fixed,float,reference,price_source
12,q1_26,81.22,NaN,81.22,[ICE]
26,q2_26,77.53,NaN,77.53,[ICE]
40,q3_26,75.48,NaN,75.48,[ICE]
54,q4_26,81.12,83.31,82.69,"[ICE, PFC]"
68,q1_27,NaN,84.03,84.03,[PFC]
82,q2_27,NaN,84.79,84.79,[PFC]
96,q3_27,NaN,85.56,85.56,[PFC]
110,q4_27,NaN,86.32,86.32,[PFC]
124,q1_28,NaN,87.10,87.10,[PFC]
138,q2_28,NaN,87.89,87.89,[PFC]


## PFC input: `use_pfc=True` sources the floating portion from the hourly PFC

`run_asof(..., use_pfc=True)` replaces the **floating** (still-open) portion of every leg
with the hourly PFC's average over that leg's own delivery window, instead of the last
futures close held flat. The **fixed** (already-observed) portion is always futures-only.

`q4_26` mixes a fully-settled leg (`Y`), a partially-open leg (`Q`), and fully-open legs
(`M1`/`M2`/`M3`), so the comparison below shows all three outcomes side by side.

In [17]:
PFC_TARGET_QUARTER = "q1_27"

futures_only = run_asof(FULL_ASOF, df_power, df_gas, df_eua, PFC_TARGET_QUARTER, use_pfc=False)
with_pfc = run_asof(FULL_ASOF, df_power, df_gas, df_eua, PFC_TARGET_QUARTER, use_pfc=True)

power_cols = ["strip", "fixed", "float", "reference", "price_source"]
comparison = with_pfc["df"][with_pfc["df"]["commodity"] == "power"][power_cols].merge(
    futures_only["df"][futures_only["df"]["commodity"] == "power"][power_cols],
    on="strip", suffixes=(" (use_pfc)", " (futures only)"),
)
comparison

,strip,fixed (use_pfc),float (use_pfc),reference (use_pfc),price_source (use_pfc),fixed (futures only),float (futures only),reference (futures only),price_source (futures only)
0,Y,63.03,70.91,68.84,"[OMIP, PFC]",63.03,70.90,68.84,[OMIP]
1,Q,NaN,87.55,87.55,[PFC],NaN,87.75,87.75,[OMIP]
2,M1,NaN,99.84,99.84,[PFC],NaN,100.81,100.81,[OMIP]
3,M2,NaN,91.31,91.31,[PFC],NaN,87.75,87.75,"[OMIP, propagated from Q]"
4,M3,NaN,71.87,71.87,[PFC],NaN,87.75,87.75,"[OMIP, propagated from Q]"
5,total,NaN,83.45,82.93,"[OMIP, PFC]",NaN,85.50,84.98,"[OMIP, OMIP, propagated from Q]"


In [22]:
df = with_pfc['df']
df[df['commodity'] == 'power']


,quarter,commodity,strip,fixed,float,reference,pct_open,price_source,primary_direct
0,q1_27,power,Y,63.03,70.91,68.84,0.74,"[OMIP, PFC]",False
1,q1_27,power,Q,NaN,87.55,87.55,1.00,[PFC],False
2,q1_27,power,M1,NaN,99.84,99.84,1.00,[PFC],False
3,q1_27,power,M2,NaN,91.31,91.31,1.00,[PFC],False
4,q1_27,power,M3,NaN,71.87,71.87,1.00,[PFC],False
5,q1_27,power,total,NaN,83.45,82.93,0.93,"[OMIP, PFC]",False


In [24]:
df = futures_only['df']
df[df['commodity'] == 'power']

,quarter,commodity,strip,fixed,float,reference,pct_open,price_source,primary_direct
0,q1_27,power,Y,63.03,70.90,68.84,0.74,[OMIP],True
1,q1_27,power,Q,NaN,87.75,87.75,1.00,[OMIP],True
2,q1_27,power,M1,NaN,100.81,100.81,1.00,[OMIP],True
3,q1_27,power,M2,NaN,87.75,87.75,1.00,"[OMIP, propagated from Q]",False
4,q1_27,power,M3,NaN,87.75,87.75,1.00,"[OMIP, propagated from Q]",False
5,q1_27,power,total,NaN,85.50,84.98,0.93,"[OMIP, OMIP, propagated from Q]",False
